# Data Engineering and PySpark

**Case study:** ETL pipeline for retail orders. Synthetic data only. Run cells in order.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("data") if Path("data/customers_synthetic.csv").exists() else Path(".")
customers = pd.read_csv(DATA / "customers_synthetic.csv")
orders = pd.read_csv(DATA / "orders_synthetic.csv", parse_dates=["order_date"])
print("Customers:", customers.shape, "Orders:", orders.shape)
display(customers.head())

## Data quality and exploratory checks

In [ ]:
assert customers.customer_id.is_unique
assert orders.order_id.is_unique
assert customers.churned.isin([0,1]).all()
print(customers.isna().sum().to_string())
print("Churn rate:", round(customers.churned.mean(),3))
customers.groupby("plan").churned.agg(["mean","count"]).plot(kind="bar",subplots=True,figsize=(9,3),title=["Churn rate","Customers"]); plt.tight_layout(); plt.show()

## Define data-quality checks and a deterministic pandas reference

In [ ]:
assert orders.order_id.is_unique
assert orders.quantity.gt(0).all()
assert orders.unit_price.ge(0).all()
completed=orders.loc[orders.status.eq("completed")].copy()
completed["revenue"]=completed.quantity*completed.unit_price
expected=completed.groupby("category").revenue.sum().sort_index()
print(expected.round(2))

## PySpark local ETL (requires Java and PySpark)

If Spark cannot initialize on your environment, use local Jupyter with a compatible Java runtime.

In [ ]:
from pyspark.sql import SparkSession,functions as F
spark=SparkSession.builder.master("local[2]").appName("retail-etl-study").getOrCreate()
sdf=spark.read.option("header",True).option("inferSchema",True).csv(str(DATA / "orders_synthetic.csv"))
assert sdf.count()==len(orders)
curated=(sdf.filter(F.col("status")=="completed").withColumn("revenue",F.col("quantity")*F.col("unit_price")))
summary=curated.groupBy("category").agg(F.round(F.sum("revenue"),2).alias("revenue")).orderBy("category")
summary.show()
actual={r["category"]:r["revenue"] for r in summary.collect()}
for category,value in expected.items():assert abs(actual[category]-round(float(value),2))<.011
print("Spark matches pandas reference")

## Join and window functions

In [ ]:
from pyspark.sql.window import Window
customer_sdf=spark.read.option("header",True).option("inferSchema",True).csv(str(DATA / "customers_synthetic.csv"))
joined=curated.join(customer_sdf.select("customer_id","region"),on="customer_id",how="left")
region_sales=joined.groupBy("region").agg(F.round(F.sum("revenue"),2).alias("revenue"))
ranked=region_sales.withColumn("rank",F.dense_rank().over(Window.orderBy(F.desc("revenue"))))
ranked.show()
spark.stop()

## Reflection

1. What was your baseline? 2. What could go wrong in deployment? 3. Which results require human review?

Complete `Practice_Questions.md` before reading `Solutions.md`.